# Part 4: Hybrid Recommendation System & Evaluation
In previous notebook, two systems independent recommendation:
* **Content-Based Filtering:** Recommend based on genres of movies. Very good to solve **Cold Start** problem but easy to meet **Circular Recommendation** problem
* **Collaborative Filtering (Item-Item):** Recommend based on group behaviour. Recommend many types of movies and understanding "Hidden preferences"

**Goal of this notebook:** Build a Hybrid system to combine the power of two models above

In [1]:
%load_ext autoreload
%autoreload 2

## Import libraries

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import joblib

import sys
sys.path.append('..')
from src.utils import load_movie_mapping,get_items_rated_by_user,get_movie_title_by_id
from src.recommender import ContentBasedFiltering, NBCF

## 1.Load data and pre-train models

### 1.1.Train-test data

In [5]:
ratings_train=pd.read_csv('../data/processed/ratings_a_train.csv',header=0)
ratings_train

,user_id,movie_id,rating,timestamp
0,0,0,5,874965758
1,0,1,3,876893171
2,0,2,4,878542960
3,0,3,3,876893119
4,0,4,3,889751712
...,...,...,...,...
90565,942,1046,2,875502146
90566,942,1073,4,888640250
90567,942,1187,3,888640250
90568,942,1227,3,888640275


In [7]:
ratings_test=pd.read_csv('../data/processed/ratings_a_test.csv',header=0)
ratings_test

,user_id,movie_id,rating,timestamp
0,0,19,4,887431883
1,0,32,4,878542699
2,0,60,4,878542420
3,0,116,3,874965739
4,0,154,2,878542201
...,...,...,...,...
9425,942,231,4,888639867
9426,942,355,4,888639598
9427,942,569,1,888640125
9428,942,807,4,888639868


### 1.2.Load models

**Content-Based model**

In [8]:
cb_model=joblib.load('../models/cb_model.pkl')

In [9]:
cb_model

**Collaborative Filtering (Item-Item) model**

In [10]:
cf_model=joblib.load('../models/cf_model.pkl')

## 2.Build Hybrid System

We will encapsulate hybrid logic in a class, use function:

$$\hat{y}_{hybrid}=\alpha \cdot \hat{y}_{cb}+(1-\alpha) \cdot \hat{y}_{cf}$$

**Where:**
* $\alpha \in [0,1]$

In [ ]:
class HybridRecommender:
    def __init__(self,cb_model,cf_model,alpha=0.5):
        self.cb_model=cb_model
        self.cf_model=cf_model
        self.alpha=alpha
    def predict_test(self,test):
        cb_predict=self.cb_model.predict(test)
        cf_predict=self.cf_model.predict_test(test)
        return self.alpha*cb_predict+(1-self.alpha)*cf_predict
    def fit(self,ratings_data):
        self.ratings_data=ratings_data
    def recommend_for_user(self,user_id,top=10):
        rated_items,_=get_items_rated_by_user(self.ratings_data,user_id)
        rated_items=rated_items.tolist()
        
        n_items=self.cb_model.Yhat.shape[0]
        hybrid_scores=np.zeros(n_items)
        cb_scores=self.cb_model.Yhat[:,user_id]
        for i in range(n_items):
            if i in rated_items:
                hybrid_scores[i]=-1e9
            else:
                cf_score=self.cf_model.predict(user_id,i,normalize=0)
                cb_score=cb_scores[i]
                hybrid_scores[i]=self.alpha*cb_score+(1-self.alpha)*cf_score
        recommend_items=np.argsort(hybrid_scores)[-top:][::-1]
        return recommend_items
    def RMSE(self,ratings_data,prediction):
        y_true=ratings_data[:,2]
        return np.sqrt(mean_squared_error(y_true,prediction))  